# Phase 1 : ancre BART-large (MEDIQA 2021)

Runtime : **GPU L4** (ou A100), via *Exécution > Modifier le type d'exécution*. Un T4 est refusé : il n'a pas de bf16 natif. Durée : environ 45 à 60 minutes. Toute la logique est dans le dépôt.

- **« Tout exécuter » s'arrête à la première erreur**, et l'erreur affichée est la vraie. Le journal complet de chaque étape est gardé sur Drive (`faithqs/state/logs/`).
- La **vérification** (2-3 min) entraîne un BART minuscule et calcule toutes les métriques sur ce GPU avant les runs longs.
- **Google Drive** garde l'état du protocole (taux choisi, **registre du test**) et une copie des runs **sans les poids**. Relancer le notebook ne peut ni perdre la sélection, ni rouvrir l'accès au test.
- Chaque étape **reprend** là où elle s'est arrêtée : un run terminé n'est pas refait, un run interrompu est mis de côté puis relancé.
- La cellule « État » dit à tout moment ce qui est fait.

In [ ]:
REPO_URL = "https://github.com/desire-del/faithful-question-summarization.git"
REF = "main"
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['FAITHQS_STATE_DIR'] = '/content/drive/MyDrive/faithqs/state'  # ledger + protocol state
os.makedirs(os.environ['FAITHQS_STATE_DIR'], exist_ok=True)
%cd /content
# Never delete an existing clone: update it instead.
!if [ -d repo/.git ]; then git -C repo fetch -q origin && git -C repo checkout -q $REF && git -C repo pull -q --ff-only origin $REF; else git clone -q --branch $REF $REPO_URL repo; fi
%cd /content/repo
!git log --oneline -1
# Exact versions from uv.lock, except the torch/CUDA stack (Colab's build matches its driver).
!pip install -q uv
!uv export --frozen --extra train --extra eval --no-hashes --no-emit-project -o /content/req_lock.txt
!grep -v -E '^(torch|triton|nvidia-|cuda-|torchvision|torchaudio)' /content/req_lock.txt > /content/req.txt
# Unused here, and built for one exact torch: a mismatch breaks transformers imports. torchao: peft rejects it.
!pip uninstall -y -q torchao torchvision torchaudio
!pip install -q -r /content/req.txt
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Helpers: run a step with its full log on Drive; stop 'Run all' on the first failure.
import codecs, datetime, os, pathlib, shutil, subprocess, sys
STATE = pathlib.Path(os.environ['FAITHQS_STATE_DIR'])
RUNS = pathlib.Path('/content/faithqs-runs')  # ephemeral: holds the model weights
WEIGHTS = {'model', 'checkpoints'}

def backup_runs():
    """Copy everything except weights (manifests, predictions, scores) to Drive."""
    if not RUNS.exists():
        return
    for path in RUNS.rglob('*'):
        rel = path.relative_to(RUNS)
        if path.is_file() and not WEIGHTS & set(rel.parts):
            target = STATE / 'runs' / rel
            if not target.exists() or target.stat().st_size != path.stat().st_size:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)

def run(name, *cmd):
    log = STATE / 'logs' / f"{datetime.datetime.now():%Y%m%d-%H%M%S}_{name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    with log.open('wb') as f:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                env=dict(os.environ, PYTHONUNBUFFERED='1'))
        try:
            while chunk := proc.stdout.read1(65536):
                f.write(chunk)
                f.flush()
                sys.stdout.write(decoder.decode(chunk))
                sys.stdout.flush()
        except KeyboardInterrupt:
            proc.terminate()
            raise
        finally:
            backup_runs()
    if proc.wait() != 0:
        raise RuntimeError(f"step '{name}' failed (exit code {proc.returncode}). "
                           f"Full log on Drive: {log}")
    print(f"\n[ok] {name} (log: {log})")

PROTOCOL = ['python', 'scripts/run_sft_protocol.py', '--env', 'colab', '--model', 'bart_large',
            '--sft-config', 'configs/sft/bart_large.yaml', '--tag', 'bart', '--stage']

In [ ]:
# Rebuild the splits and verify them against the versioned manifest
run('data_fetch', 'faithqs', 'data', 'fetch')

In [ ]:
# Vérification (2-3 min) : GPU, bf16 natif, puis entraînement + génération + toutes les métriques
# sur un BART minuscule. Si elle échoue, rien de long n'a été lancé.
run('preflight', 'python', 'scripts/preflight.py', '--env', 'colab')

In [ ]:
# État : ce qui est fait (relance cette cellule quand tu veux)
import json
sel = STATE / 'reports/sft/bart_lr_selection.json'
print('lr selection :', json.loads(sel.read_text())['chosen_lr'] if sel.exists() else 'not done yet')
for run_dir in sorted((STATE / 'runs').glob('bart_*')):
    scores = sorted(p.name for p in (run_dir / 'scores').glob('*.summary.json')) if (run_dir / 'scores').exists() else []
    weights = 'model' if (RUNS / run_dir.name / 'model').exists() else 'no weights here'
    print(f"{run_dir.name:40s} {weights:16s} {scores}")
ledger = STATE / 'test_ledger.jsonl'
print('test ledger  :', ledger.read_text().strip() if ledger.exists() else 'empty (test never accessed)')
logs = sorted((STATE / 'logs').glob('*.log'))[-5:]
print('recent logs  :', *[p.name for p in logs] or ['none'], sep='\n  ')

In [ ]:
# 1) Learning-rate search (seed 0), choice on dev ROUGE-L. Ends with 'chosen lr: ...'
run('search', *PROTOCOL, 'search')

In [ ]:
# 2) Seeds 0-2 at the chosen rate (finished runs are reused) + dev predictions (greedy, beam4)
run('seeds', *PROTOCOL, 'seeds')

In [ ]:
# 3) Dev table
run('table', *PROTOCOL, 'table')

## ⚠️ Accès au test, une seule fois
La cellule suivante consomme l'accès `bart_reproduction` (PROTOCOL §2). Elle vérifie d'abord que les 3 modèles existent (sinon elle s'arrête **sans** toucher au test), puis journalise l'accès sur Drive et le ferme. Une seconde tentative est refusée.

In [ ]:
# 4) BART reproduction on the official MEDIQA 2021 test set (logged access)
run('test', *PROTOCOL, 'test')

In [ ]:
# 5) Artifacts: run manifests, predictions, scores, protocol reports and the test ledger (no weights)
import zipfile
backup_runs()
archive = pathlib.Path('/content/bart_anchor_artifacts.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in (STATE / 'runs').rglob('*'):
        rel = path.relative_to(STATE / 'runs')
        if path.is_file() and '.incomplete-' not in rel.parts[0]:  # interrupted runs stay on Drive
            z.write(path, 'runs/' + str(rel))
    for path in (STATE / 'reports').rglob('*'):
        if path.is_file():
            z.write(path, 'reports/' + str(path.relative_to(STATE / 'reports')))
    if (STATE / 'test_ledger.jsonl').exists():
        z.write(STATE / 'test_ledger.jsonl', 'reports/test_ledger.jsonl')
print(archive, round(archive.stat().st_size / 1e6, 2), 'MB')
shutil.copy2(archive, STATE.parent / archive.name)
from google.colab import files
files.download(str(archive))

In [ ]:
# 6) Summary to paste back
rep_dir = STATE / 'reports/sft'
print((rep_dir / 'bart_dev.md').read_text())
rep = json.loads((rep_dir / 'bart_test_reproduction.json').read_text())
print('chosen lr', rep['lr'])
print(json.dumps(rep['vs_focusmed_bart'], indent=1))